In [ ]:
! pip install evidently

In [11]:
import evidently
evidently.__version__

'0.6.6'

In [7]:
import joblib
import pandas as pd
from steps.data_preprocessing import Cleaner
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset , DataQualityPreset , TargetDriftPreset 
from evidently import ColumnMapping
import warnings
warnings.filterwarnings("ignore")


In [8]:
model = joblib.load('models/model.pkl')

reference = pd.read_csv("data/train.csv")
current = pd.read_csv("data/test.csv")
production = pd.read_csv("data/production.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'models/model.pkl'

  Cloning https://github.com/evidentlyai/evidently.git to c:\users\parthiban\appdata\local\temp\pip-req-build-504einmk
  Resolved https://github.com/evidentlyai/evidently.git to commit 7eaf5f450f6e6f8573f8e801f84dfa4d916a6ab9
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached evidently-0.6.6-py3-none-any.whl.metadata (11 kB)
  Using cached litestar-2.15.1-py3-none-any.whl.metadata (109 kB)
  Using cached iterative_telemetry-0.0.10-py3-none-any.whl.metadata (4.1 kB)
  Using cached dynaconf-3.2.10-py2.py3-none-any.whl.metadata (9.1 kB)


  Running command git clone --filter=blob:none --quiet https://github.com/evidentlyai/evidently.git 'C:\Users\Parthiban\AppData\Local\Temp\pip-req-build-504einmk'
ERROR: Exception:
Traceback (most recent call last):
  File "D:\Users\Lib\site-packages\pip\_internal\cli\base_command.py", line 180, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "D:\Users\Lib\site-packages\pip\_internal\cli\req_command.py", line 245, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\Users\Lib\site-packages\pip\_internal\commands\install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "D:\Users\Lib\site-packages\pip\_internal\resolution\resolvelib\resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "D:\Users\Lib\site-packages\pip\_vendor\resolvelib\resolvers.py", line 546, in resolv

In [ ]:
cleaner = Cleaner()
reference = cleaner.clean_data(reference)
reference['prediction'] = model.predict(reference.iloc[:,:-1])

current = cleaner.clean_data(current)
current['predictions'] = model.predict(current.iloc[:,:-1])

production = cleaner.clean_data(production)
production['prediction'] = model.predict(current.iloc[:,:-1])

In [ ]:
target = "Result"
prediction = 'prediction'
numerical_features = ['Age','AnnualPremium','HasDrivingLicense','RegionID','Switch']
categorical_features = ['Gender','PastAccident']
column_mapping = ColumnMapping()

column_mapping.target = target
column_mapping.prediction = prediction
column_mapping.numerical_features = numerical_features
column_mapping.categorical_features = categorical_features

In [ ]:
data_drift_report = Report(metrics = [
    DataDriftPreset(),
    DataQualityPreset(),
    TargetQualitypreset()
])

data_drift_report.run(reference_data = reference, current_data = current , column_mapping = column_mapping)
data_drift_report
data_drift_report.save_html("test_drift.html")
